In [21]:
import re
import fitz  # PyMuPDF
from dataclasses import dataclass


@dataclass
class PDFPage:
    text: str
    page_number: int
    source: str


In [ ]:
pdf_path = "OJ_L_202401689_EN_TXT.pdf"
with fitz.open(pdf_path) as doc:
    page = doc[0]
    try:    
        text = page.get_text("text")
    except Exception:
        text = page.get_text("rawtext")
print(text)

REGULATION (EU) 2024/1689 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL
of 13 June 2024
laying down harmonised rules on artificial intelligence and amending Regulations (EC) No 300/2008, 
(EU) No 167/2013, (EU) No 168/2013, (EU) 2018/858, (EU) 2018/1139 and (EU) 2019/2144 and 
Directives 2014/90/EU, (EU) 2016/797 and (EU) 2020/1828 (Artificial Intelligence Act)
(Text with EEA relevance)
THE EUROPEAN PARLIAMENT AND THE COUNCIL OF THE EUROPEAN UNION,
Having regard to the Treaty on the Functioning of the European Union, and in particular Articles 16 and 114 thereof,
Having regard to the proposal from the European Commission,
After transmission of the draft legislative act to the national parliaments,
Having regard to the opinion of the European Economic and Social Committee (1),
Having regard to the opinion of the European Central Bank (2),
Having regard to the opinion of the Committee of the Regions (3),
Acting in accordance with the ordinary legislative procedure (4),
Whereas:
(1)
The p

In [38]:
pdf_path = "OJ_L_202401689_EN_TXT.pdf"
with fitz.open(pdf_path) as doc:
    page = doc[0]
    try:    
        text = page.get_drawings()
    except Exception:
        text = page.get_text("rawtext")
text

[{'items': [('l',
    Point(84.35910034179688, 64.2900390625),
    Point(41.83940124511719, 64.2900390625)),
   ('l',
    Point(41.83940124511719, 64.2900390625),
    Point(41.83940124511719, 63.780029296875)),
   ('l',
    Point(41.83940124511719, 63.780029296875),
    Point(84.35910034179688, 63.780029296875))],
  'type': 'f',
  'even_odd': True,
  'fill_opacity': 1.0,
  'fill': (0.13669031858444214, 0.12195010483264923, 0.1252918243408203),
  'rect': Rect(41.83940124511719, 63.780029296875, 84.35910034179688, 64.2900390625),
  'seqno': 27,
  'layer': '',
  'closePath': None,
  'color': None,
  'width': None,
  'lineCap': None,
  'lineJoin': None,
  'dashes': None,
  'stroke_opacity': None},
 {'items': [('l',
    Point(411.7040100097656, 64.2900390625),
    Point(84.35910034179688, 64.2900390625)),
   ('l',
    Point(84.35910034179688, 64.2900390625),
    Point(84.35910034179688, 63.780029296875)),
   ('l',
    Point(84.35910034179688, 63.780029296875),
    Point(411.7040100097656, 6

In [15]:

# Parameters for detecting horizontal border lines
MIN_LINE_WIDTH_RATIO = 0.5   # Line must span at least this fraction of page width
MAX_LINE_HEIGHT      = 3.0   # Max height (pts) for a drawing to be treated as a line
TOP_ZONE_RATIO       = 0.35  # Top border must be in the top N% of the page
BOTTOM_ZONE_RATIO    = 0.65  # Bottom border must be in the bottom (1-N)% of the page


def find_content_bounds(page,
                        min_line_width_ratio=MIN_LINE_WIDTH_RATIO,
                        max_line_height=MAX_LINE_HEIGHT,
                        top_zone_ratio=TOP_ZONE_RATIO,
                        bottom_zone_ratio=BOTTOM_ZONE_RATIO):
    """
    Scan the drawings on a PDF page to locate:
      - The lowest horizontal line in the top zone  → top content boundary
      - The highest horizontal line in the bottom zone → bottom content boundary

    Returns (top_y, bottom_y).  If a boundary is not found the corresponding
    value is set to the page top (0) or page bottom (page height).
    """
    page_rect   = page.rect
    page_width  = page_rect.width
    page_height = page_rect.height
    min_width   = page_width * min_line_width_ratio

    top_zone_max    = page_height * top_zone_ratio
    bottom_zone_min = page_height * bottom_zone_ratio

    top_candidates    = []   # lines in the top zone
    bottom_candidates = []   # lines in the bottom zone

    for drawing in page.get_drawings():
        rect   = drawing["rect"]
        width  = rect.width
        height = rect.height
        y_mid  = (rect.y0 + rect.y1) / 2

        if width >= min_width and height <= max_line_height:
            if y_mid <= top_zone_max:
                top_candidates.append(y_mid)
            elif y_mid >= bottom_zone_min:
                bottom_candidates.append(y_mid)

    # Use the lowest line in the top zone as the top border
    top_y    = max(top_candidates)    if top_candidates    else page_rect.y0
    # Use the highest line in the bottom zone as the bottom border
    bottom_y = min(bottom_candidates) if bottom_candidates else page_rect.y1

    return top_y, bottom_y


pdf_path = "OJ_L_202401689_EN_TXT.pdf"
with fitz.open(pdf_path) as doc:
    page = doc[0]
    page_height = page.rect.height

    # ── Diagnostics: show every drawing that qualifies as a horizontal line ──
    print(f"Page size: {page.rect.width:.1f} x {page_height:.1f} pts\n")
    all_lines = [
        (drawing["rect"], (drawing["rect"].y0 + drawing["rect"].y1) / 2)
        for drawing in page.get_drawings()
        if drawing["rect"].width >= page.rect.width * MIN_LINE_WIDTH_RATIO
        and drawing["rect"].height <= MAX_LINE_HEIGHT
    ]
    print(f"Detected {len(all_lines)} horizontal border line(s):")
    for rect, y_mid in sorted(all_lines, key=lambda x: x[1]):
        print(f"  y_mid={y_mid:.2f}  rect={rect}  (page {y_mid/page_height*100:.1f}% from top)")

    # ── Detect top / bottom bounds and extract content ──
    print()
    top_y, bottom_y = find_content_bounds(page)
    print(f"Top content boundary   : y = {top_y:.2f} pts  ({top_y/page_height*100:.1f}% from top)")
    print(f"Bottom content boundary: y = {bottom_y:.2f} pts  ({bottom_y/page_height*100:.1f}% from top)")

    clip = fitz.Rect(0, top_y, page.rect.width, bottom_y)
    content_text = page.get_text("text", clip=clip)
    print("\n--- Extracted content between border lines ---\n")
    print(content_text)


Page size: 595.3 x 841.9 pts

Detected 2 horizontal border line(s):
  y_mid=64.04  rect=Rect(84.35910034179688, 63.780029296875, 411.7040100097656, 64.2900390625)  (page 7.6% from top)
  y_mid=806.43  rect=Rect(41.83940124511719, 806.1735229492188, 496.7430114746094, 806.6837158203125)  (page 95.8% from top)

Top content boundary   : y = 64.04 pts  (7.6% from top)
Bottom content boundary: y = 806.43 pts  (95.8% from top)

--- Extracted content between border lines ---

REGULATION (EU) 2024/1689 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL
of 13 June 2024
laying down harmonised rules on artificial intelligence and amending Regulations (EC) No 300/2008, 
(EU) No 167/2013, (EU) No 168/2013, (EU) 2018/858, (EU) 2018/1139 and (EU) 2019/2144 and 
Directives 2014/90/EU, (EU) 2016/797 and (EU) 2020/1828 (Artificial Intelligence Act)
(Text with EEA relevance)
THE EUROPEAN PARLIAMENT AND THE COUNCIL OF THE EUROPEAN UNION,
Having regard to the Treaty on the Functioning of the European Union, and

In [20]:
import fitz  # PyMuPDF
from operator import itemgetter

def extract_main_content_with_footnote_detection(pdf_path: str) -> str:
    """
    Extracts the main content from a PDF by dynamically identifying content
    boundaries based on horizontal line drawings, including a shorter line
    that separates footnotes.

    Args:
        pdf_path: The file path to the PDF document.

    Returns:
        A string containing the extracted text from all pages.
    """
    full_text = ""
    try:
        doc = fitz.open(pdf_path)
    except Exception as e:
        return f"Error opening PDF file: {e}"

    for page_num in range(len(doc)):
        page = doc.load_page(page_num)
        page_width = page.rect.width
        
        # --- Identify horizontal lines on the page ---
        drawings = page.get_drawings()
        h_lines = []
        for p in drawings:
            for item in p["items"]:
                if item[0] == "l":  # It's a line
                    p1 = item[1]
                    p2 = item[2]
                    # Check if it's a horizontal line
                    if abs(p1.y - p2.y) < 1:
                        width = abs(p1.x - p2.x)
                        h_lines.append({"y": p1.y, "width": width})

        if not h_lines:
            # If no lines, extract text from the full page as a fallback
            full_text += f"--- Page {page_num + 1} (no lines found) ---\n"
            full_text += page.get_text("text") + "\n\n"
            continue
            
        # Sort lines by their y-coordinate
        h_lines.sort(key=itemgetter("y"))

        # --- Determine Content Boundaries ---
        # Top boundary is the first long line (header separator)
        top_line = next((line for line in h_lines if line["width"] > 400), None)
        y0 = top_line["y"] if top_line else 80.0

        # Default bottom boundary is the last long line (footer separator)
        bottom_line = next((line for line in reversed(h_lines) if line["width"] > 400), None)
        y1 = bottom_line["y"] if bottom_line else 790.0

        # Find a potential footnote separator line (short line)
        # It must be below the header and above the footer.
        footnote_separator = next(
            (line for line in reversed(h_lines) 
             if 50 < line["width"] < 200 and y0 < line["y"] < y1), 
            None
        )
        
        # If a footnote separator is found, it becomes the new bottom boundary
        if footnote_separator:
            y1 = footnote_separator["y"]

        # Define the clipping rectangle and extract text
        clip_rect = fitz.Rect(0, y0, page_width, y1)
        page_text = page.get_text("text", clip=clip_rect).strip()
        print(page_text)
        break
        
        full_text += f"--- Page {page_num + 1} ---\n"
        full_text += page_text
        full_text += "\n\n"

    doc.close()
    return full_text

# --- Execution ---
pdf_file = "OJ_L_202401689_EN_TXT.pdf"
extracted_text = extract_main_content_with_footnote_detection(pdf_file)


In [18]:
extracted_text

'--- Page 1 ---\n\n\n--- Page 2 ---\n\n\n--- Page 3 ---\n\n\n--- Page 4 ---\n\n\n--- Page 5 ---\n\n\n--- Page 6 ---\n\n\n--- Page 7 ---\n\n\n--- Page 8 ---\n\n\n--- Page 9 ---\n\n\n--- Page 10 ---\n\n\n--- Page 11 ---\n\n\n--- Page 12 ---\n\n\n--- Page 13 ---\n\n\n--- Page 14 ---\n\n\n--- Page 15 ---\n\n\n--- Page 16 ---\n\n\n--- Page 17 ---\n\n\n--- Page 18 ---\n\n\n--- Page 19 ---\n\n\n--- Page 20 ---\n\n\n--- Page 21 ---\n\n\n--- Page 22 ---\n\n\n--- Page 23 ---\n\n\n--- Page 24 ---\n\n\n--- Page 25 ---\n\n\n--- Page 26 ---\n\n\n--- Page 27 ---\n\n\n--- Page 28 ---\n\n\n--- Page 29 ---\n\n\n--- Page 30 ---\n\n\n--- Page 31 ---\n\n\n--- Page 32 ---\n\n\n--- Page 33 ---\n\n\n--- Page 34 ---\n\n\n--- Page 35 ---\n\n\n--- Page 36 ---\n\n\n--- Page 37 ---\n\n\n--- Page 38 ---\n\n\n--- Page 39 ---\n\n\n--- Page 40 ---\n\n\n--- Page 41 ---\n\n\n--- Page 42 ---\n\n\n--- Page 43 ---\n\n\n--- Page 44 ---\n\n\n--- Page 45 ---\n\n\n--- Page 46 ---\n\n\n--- Page 47 ---\n\n\n--- Page 48 ---\n\n\n

In [29]:
pages = []
pdf_path = "OJ_L_202401689_EN_TXT.pdf"
with fitz.open(pdf_path) as doc:
    for page_index in range(len(doc)):
        page = doc[page_index]

        try:
            text = page.get_text("blocks")
            text = page.get_text("text")
        except Exception:
            try:
                text = page.get_text("rawtext")
            except Exception:
                continue

        # Handle encoding errors by encoding/decoding with replacement
        try:
            text = text.encode("utf-8", errors="replace").decode("utf-8")
        except Exception:
            continue

        if len(text.strip()) < 20:
            continue

        pages.append(
            PDFPage(
                text=text,
                page_number=page_index + 1,
                source=pdf_path,
            )
        )
print(f"Loaded {len(pages)} pages from {pdf_path}")

Loaded 144 pages from OJ_L_202401689_EN_TXT.pdf


In [33]:
pdf_path = "OJ_L_202401689_EN_TXT.pdf"
with fitz.open(pdf_path) as doc:

    page = doc[0]

    try:
        text = page.get_text("text")
    except Exception:
        text = page.get_text("rawtext")

text

'REGULATION (EU) 2024/1689 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL\nof 13 June 2024\nlaying down harmonised rules on artificial intelligence and amending Regulations (EC) No 300/2008, \n(EU) No 167/2013, (EU) No 168/2013, (EU) 2018/858, (EU) 2018/1139 and (EU) 2019/2144 and \nDirectives 2014/90/EU, (EU) 2016/797 and (EU) 2020/1828 (Artificial Intelligence Act)\n(Text with EEA relevance)\nTHE EUROPEAN PARLIAMENT AND THE COUNCIL OF THE EUROPEAN UNION,\nHaving regard to the Treaty on the Functioning of the European Union, and in particular Articles 16 and 114 thereof,\nHaving regard to the proposal from the European Commission,\nAfter transmission of the draft legislative act to the national parliaments,\nHaving regard to the opinion of the European Economic and Social Committee (1),\nHaving regard to the opinion of the European Central Bank (2),\nHaving regard to the opinion of the Committee of the Regions (3),\nActing in accordance with the ordinary legislative procedure (4),\nWhe

In [39]:
"""
extract_main_text.py
====================
Extracts the main body text from EU Official Journal PDFs (OJ L series)
by using PyMuPDF's get_drawings() to identify structural separator lines:

  ┌─────────────────────────────┐
  │  Header area                │  ← above HEADER_LINE_Y (≈63.78)
  ├─────────────────────────────┤  ← HEADER separator line (static)
  │                             │
  │  MAIN TEXT BODY             │  ← what we want to extract
  │                             │
  ├─────────────────────────────┤  ← FOOTNOTE separator line (dynamic y)
  │  Footnotes                  │
  ├─────────────────────────────┤  ← FOOTER separator line (static, ≈806.17)
  │  Footer area                │
  └─────────────────────────────┘

Key observations from get_drawings():
--------------------------------------
1.  ORDER: Drawings are returned in PDF content-stream order (i.e., seqno
    order), NOT necessarily top-to-bottom visual order. On page 1, the
    footnote line (seqno=37, y≈748) appears AFTER the footer line
    (seqno=34-35, y≈806). Always sort by y0 if you need visual order.

2.  HEADER LINE (static):
      - y0 ≈ 63.78 (tolerance ±2 px)
      - Spans full text column: x0 ≈ 41.84, x1 ≈ 553.44
      - Often split into 2–3 collinear segments (same y0, abutting x-ranges)
      - height ≈ 0.51 (thin hairline rule)

3.  FOOTER LINE (static):
      - y0 ≈ 806.17 (tolerance ±2 px)
      - Same thin hairline, same full-width span
      - Also sometimes split into 2 segments

4.  FOOTNOTE SEPARATOR LINE (dynamic):
      - y0 varies per page (e.g., 748.69 on p.1, 767.40 on p.2, 589.83 on p.3)
      - Always a SHORT line: x0 ≈ 67.41, x1 ≈ 118.38 (width ≈ 51 px)
      - Same thin height ≈ 0.51
      - NOT present on pages with no footnotes

Strategy
---------
  - Find all hairline rules (height < 2 px, filled, spanning ≥ 10 px wide)
  - Merge collinear segments (same y0 ± 1 px, abutting x ranges)
  - Classify merged lines by y0 and width:
      * HEADER  : y0 < page_height/2 AND total_width > 400
      * FOOTER  : y0 > page_height/2 AND total_width > 400
      * FOOTNOTE: total_width < 100 (short rule, any y)
  - Clip text blocks to the window (header_y1 … footnote_y0 or footer_y0)
"""

import fitz  # PyMuPDF


# ── Tolerances & thresholds ────────────────────────────────────────────────────
HAIRLINE_MAX_HEIGHT = 2.0      # px — max height to be considered a rule/line
HAIRLINE_MIN_WIDTH  = 10.0     # px — ignore tiny decorative marks
FULL_WIDTH_THRESHOLD = 400.0   # px — header/footer lines span most of the page
FOOTNOTE_MAX_WIDTH   = 100.0   # px — footnote rule is short (~51 px)
Y_SNAP_TOLERANCE     = 1.5     # px — treat lines at same y as one line


def _collect_hairlines(drawings):
    """Return raw hairline segments from a page's get_drawings() output."""
    lines = []
    for d in drawings:
        r = d["rect"]
        # Only filled rectangles that are extremely thin
        if d.get("type") != "f":
            continue
        if r.height > HAIRLINE_MAX_HEIGHT and r.width > HAIRLINE_MAX_HEIGHT:
            continue  # not a thin rule in either direction
        if max(r.width, r.height) < HAIRLINE_MIN_WIDTH:
            continue  # too small to matter
        lines.append(r)
    return lines


def _merge_collinear(segments):
    """
    Merge collinear segments (same y0 within tolerance, abutting x-ranges)
    into a list of (y0, x0, x1) tuples representing full logical lines.
    """
    if not segments:
        return []

    # Group by rounded y0
    from collections import defaultdict
    groups = defaultdict(list)
    for r in segments:
        # Snap y0 to nearest bucket
        key = round(r.y0 / Y_SNAP_TOLERANCE) * Y_SNAP_TOLERANCE
        groups[key].append(r)

    merged = []
    for y_key, rects in groups.items():
        x0 = min(r.x0 for r in rects)
        x1 = max(r.x1 for r in rects)
        y0 = sum(r.y0 for r in rects) / len(rects)   # average y0
        merged.append((y0, x0, x1))

    return sorted(merged)   # sort top-to-bottom by y0


def classify_lines(page):
    """
    Analyse a page's drawings and return a dict:
      {
        'header_y':   float | None,   # bottom of header separator line
        'footer_y':   float | None,   # top of footer separator line
        'footnote_y': float | None,   # top of footnote separator line (dynamic)
      }
    """
    drawings = page.get_drawings()
    segments = _collect_hairlines(drawings)
    lines    = _merge_collinear(segments)

    page_mid = page.rect.height / 2.0

    header_y   = None
    footer_y   = None
    footnote_y = None

    for y0, x0, x1 in lines:
        width = x1 - x0
        if width >= FULL_WIDTH_THRESHOLD:
            if y0 < page_mid:
                # Full-width line in the top half → header separator
                header_y = y0
            else:
                # Full-width line in the bottom half → footer separator
                footer_y = y0
        elif width <= FOOTNOTE_MAX_WIDTH:
            # Short line → footnote separator
            footnote_y = y0

    return {
        "header_y":   header_y,
        "footer_y":   footer_y,
        "footnote_y": footnote_y,
    }


def extract_main_text_blocks(page, margin: float = 2.0):
    """
    Extract text blocks that fall within the main body region of the page
    (below the header line, above the footnote separator or footer line).

    Parameters
    ----------
    page    : fitz.Page
    margin  : float — extra pixels of slack when deciding if a block is
              inside the body region (handles sub-pixel rounding)

    Returns
    -------
    list of dicts with keys:
        x0, y0, x1, y1  — bounding box
        text             — plain text content
        block_no         — original block index from get_text("blocks")
    """
    bounds = classify_lines(page)

    # Determine body window
    top = (bounds["header_y"] or 0.0) + margin
    if bounds["footnote_y"] is not None:
        bottom = bounds["footnote_y"] - margin
    elif bounds["footer_y"] is not None:
        bottom = bounds["footer_y"] - margin
    else:
        bottom = page.rect.height

    body_blocks = []
    for b in page.get_text("blocks"):
        x0, y0, x1, y1, text, block_no, block_type = b
        if block_type != 0:         # 0 = text, 1 = image
            continue
        if text.strip() == "":
            continue
        # Block must be fully within the body window.
        # Note: on page 1 the doc-number/date line (e.g. "2024/1689  12.7.2024")
        # sits at y0≈72, which is BELOW the header rule (y0≈63.78), so it is
        # correctly included.  The masthead ("Official Journal…") is at y0≈40,
        # above the rule, and is correctly excluded.
        if y0 >= top and y1 <= bottom:
            body_blocks.append({
                "x0": x0, "y0": y0, "x1": x1, "y1": y1,
                "text": text,
                "block_no": block_no,
            })

    # Return in reading order (top-to-bottom, left-to-right)
    body_blocks.sort(key=lambda b: (b["y0"], b["x0"]))
    return body_blocks


def extract_main_text(page, separator: str = "\n") -> str:
    """
    Convenience wrapper: returns main body text as a single string.
    """
    blocks = extract_main_text_blocks(page)
    return separator.join(b["text"].strip() for b in blocks)


# ── Demo / smoke-test ──────────────────────────────────────────────────────────

pdf_path = "OJ_L_202401689_EN_TXT.pdf"
doc = fitz.open(pdf_path)

# for page_num in range(min(3, len(doc))):
page   = doc[0]
bounds = classify_lines(page)
blocks = extract_main_text_blocks(page)

print(f"\n{'='*70}")
print(f"PAGE 1")
print(f"  header_y   : {bounds['header_y']}")
print(f"  footer_y   : {bounds['footer_y']}")
print(f"  footnote_y : {bounds['footnote_y']}")
print(f"  body blocks: {len(blocks)}")
print(f"{'─'*70}")
for blk in blocks:
    preview = blk["text"][:120].replace("\n", " ")
    print(f"  [y={blk['y0']:.1f}] {preview}")


PAGE 1
  header_y   : 63.780029296875
  footer_y   : 806.1735229492188
  footnote_y : 748.6868896484375
  body blocks: 17
──────────────────────────────────────────────────────────────────────
  [y=72.1] 2024/1689 12.7.2024 
  [y=99.7] REGULATION (EU) 2024/1689 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL 
  [y=128.2] of 13 June 2024 
  [y=156.6] laying down harmonised rules on artificial intelligence and amending Regulations (EC) No 300/2008,  (EU) No 167/2013, (E
  [y=205.9] (Text with EEA relevance) 
  [y=238.7] THE EUROPEAN PARLIAMENT AND THE COUNCIL OF THE EUROPEAN UNION, 
  [y=266.1] Having regard to the Treaty on the Functioning of the European Union, and in particular Articles 16 and 114 thereof, 
  [y=294.5] Having regard to the proposal from the European Commission, 
  [y=322.9] After transmission of the draft legislative act to the national parliaments, 
  [y=351.2] Having regard to the opinion of the European Economic and Social Committee (1), 
  [y=379.6] Having regard t

In [44]:
text = extract_main_text(page)
print(text)

2024/1689
12.7.2024
REGULATION (EU) 2024/1689 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL
of 13 June 2024
laying down harmonised rules on artificial intelligence and amending Regulations (EC) No 300/2008, 
(EU) No 167/2013, (EU) No 168/2013, (EU) 2018/858, (EU) 2018/1139 and (EU) 2019/2144 and 
Directives 2014/90/EU, (EU) 2016/797 and (EU) 2020/1828 (Artificial Intelligence Act)
(Text with EEA relevance)
THE EUROPEAN PARLIAMENT AND THE COUNCIL OF THE EUROPEAN UNION,
Having regard to the Treaty on the Functioning of the European Union, and in particular Articles 16 and 114 thereof,
Having regard to the proposal from the European Commission,
After transmission of the draft legislative act to the national parliaments,
Having regard to the opinion of the European Economic and Social Committee (1),
Having regard to the opinion of the European Central Bank (2),
Having regard to the opinion of the Committee of the Regions (3),
Acting in accordance with the ordinary legislative procedure (4)